In [7]:
BASE_DIR = "/Users/etlevn/Documents/PycharmProjects/LLM_Trd/exp/gpt-4o VS ds-v3 20+10"

In [8]:
import os
import json
import pandas as pd
import numpy as np

In [9]:
# Theoretical (fundamental) stock value used in all experiments
FUNDAMENTAL_VALUE = 14

# Container for all experiment results
results = []

In [10]:
# Loop over each experiment directory
for exp_name in sorted(os.listdir(BASE_DIR)):
    exp_path = os.path.join(BASE_DIR, exp_name)
    if not os.path.isdir(exp_path):
        continue  # skip non-folders

    # === Load market price series ===
    try:
        mkt_path = os.path.join(exp_path, "market", "mkt.csv")
        prices = []
        with open(mkt_path, "r") as f:
            for line in f:
                if line.strip() == "" or line.lower().startswith("round"):
                    continue
                parts = line.strip().split(",")
                if len(parts) < 2:
                    continue
                price = parts[1]
                prices.append(float(price))
    except Exception as e:
        print(f"[WARNING] Failed to read mkt.csv for {exp_name}: {e}")
        continue

    # Record actual number of rounds
    actual_rounds = len(prices)-1

    # Skip if exactly 4 rounds
    if actual_rounds == 3:
        print(f"[SKIPPED] {exp_name} 3-round prac expr")
        continue

    # === Compute MSE and Amplitude ===
    mse = np.mean([(p - FUNDAMENTAL_VALUE) ** 2 for p in prices]) if prices else None
    amplitude = max(prices) - FUNDAMENTAL_VALUE if prices else None

    # === Compute Portfolio Value Variance from latest input round ===
    try:
        input_dir = os.path.join(exp_path, "input")
        round_folders = [f for f in os.listdir(input_dir) if f.startswith("r") and os.path.isdir(os.path.join(input_dir, f))]
        if not round_folders:
            raise ValueError("No input round folders found.")
        last_round = sorted(round_folders)[-1]
        r_last_input_path = os.path.join(input_dir, last_round)

        pv_list = []
        for file in os.listdir(r_last_input_path):
            if not file.endswith("_portfolio.txt"):
                continue
            with open(os.path.join(r_last_input_path, file), "r") as f:
                lines = f.readlines()
                cash, redemp = None, None
                for line in lines:
                    if "Current cash" in line:
                        cash = float(line.strip().split(":")[1])
                    elif "Expected redemption value" in line:
                        redemp = float(line.strip().split(":")[1])
                if cash is not None and redemp is not None:
                    pv_list.append(cash + redemp)

        if len(pv_list) == 0:
            raise ValueError("No portfolio values found in .txt files.")
        pv_var = np.var(pv_list)
    except Exception as e:
        print(f"[WARNING] Failed to compute PV variance for {exp_name}: {e}")
        pv_var = None

    results.append({
        "experiment": exp_name,
        "Rounds": actual_rounds,
        "MSE": round(mse, 2) if mse is not None else "",
        "Amplitude": round(amplitude, 2) if amplitude is not None else "",
        "PV_Variance": round(pv_var, 2) if pv_var is not None else ""
    })

# Convert to DataFrame
df = pd.DataFrame(results)

# Show result
df = df.sort_values("experiment").reset_index(drop=True)

[SKIPPED] 20250809_095114 3-round prac expr


In [11]:
df.to_csv(os.path.join(BASE_DIR, "anal.csv"))
df

,experiment,Rounds,MSE,Amplitude,PV_Variance
0,20250809_104123,20,0.0,0.0,0.0


In [12]:
import matplotlib.pyplot as plt

# Create output folder for plots
PLOT_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# Plot for each experiment
for exp in results:
    exp_name = exp["experiment"]
    exp_path = os.path.join(BASE_DIR, exp_name)
    mkt_path = os.path.join(exp_path, "market", "mkt.csv")

    # Load market prices and volumes
    prices, volumes = [], []
    try:
        with open(mkt_path, "r") as f:
            for line in f:
                if line.strip() == "" or line.lower().startswith("round"):
                    continue
                parts = line.strip().split(",")
                if len(parts) < 3:
                    continue  # skip malformed lines
                _, price, volume = parts[:3]
                prices.append(float(price))
                volumes.append(float(volume))
    except Exception as e:
        print(f"[ERROR] Failed to read {mkt_path}: {e}")
        continue  # skip if file read fails

    # Skip if exactly 4 rounds
    if len(prices) == 4:
        print(f"[SKIPPED] {exp_name} has exactly 4 rounds – skipping plot")
        continue

    if len(prices) == 0:
        print(f"[SKIPPED] No market data in {exp_name}")
        continue

    rounds = list(range(len(prices)))

    # Create figure and twin y-axis
    fig, ax1 = plt.subplots(figsize=(6, 4))

    # Line plot: Price
    ax1.plot(rounds, prices, color='tab:red', marker='o', label="Market Price")
    ax1.axhline(y=FUNDAMENTAL_VALUE, linestyle="--", color="gray", label="Fundamental Value")
    ax1.set_ylabel("Price", color='tab:red')
    ax1.set_ylim(0, max(30, max(prices) + 2))
    ax1.tick_params(axis='y', labelcolor='tab:red')

    # Bar plot: Volume (second y-axis)
    ax2 = ax1.twinx()
    ax2.bar(rounds, volumes, alpha=0.3, color='tab:blue', label="Volume")
    ax2.set_ylabel("Volume", color='tab:blue')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    # Title and formatting
    title = f"{exp_name} (Rounds={len(prices)-1})\nMSE={exp['MSE']} | Amp={exp['Amplitude']} | PVVar={exp['PV_Variance']}"
    plt.title(title)
    ax1.set_xlabel("Round")
    ax1.grid(True, linestyle=":")
    fig.tight_layout()

    # Save
    save_path = os.path.join(PLOT_DIR, f"{exp_name}.png")
    plt.savefig(save_path)
    plt.close()

print(f"All dual-axis plots saved to: {PLOT_DIR}")

All dual-axis plots saved to: /Users/etlevn/Documents/PycharmProjects/LLM_Trd/exp/gpt-4o VS ds-v3 20+10/plots
